# DEC / IDEC baseline for *Multi-Mechanism Blind Spots* (Section 5.3, Future Work)

This notebook adds two purpose-built deep-clustering methods — **DEC** (Xie, Girshick & Farhadi,
2016) and **IDEC** (Guo et al., 2017) — as baselines, following exactly the paper's benchmark
generator (`dataset_lib.py`, embedded below verbatim) and its N = 80-seed, k = 4 evaluation
protocol (seeds 1000–1079, the same convention used in Sections 5.4/5.5/5.6/6).

**Run order:** Runtime → Change runtime type → **GPU** (T4 is enough), then Runtime → Run all.

**What this fills in:** the paper's Future Work section currently reads *"A purpose-built
deep-clustering method (e.g. DEC/IDEC) ... were not attempted due to scope and remain Future
Work."* This notebook attempts them, at two architectures (the paper's original 2-unit
bottleneck, for direct comparability with the existing autoencoder row in Table 1, and a wider
10-unit latent space, addressing the paper's other stated gap — "a wider architecture search").

**At the end:** download `dec_idec_results.zip` and send it back (or just paste the final
printed summary table) — that's everything needed to update Section 5.3, Table 1, and the
Future Work / Limitations sections.


In [ ]:
import torch, sys
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("\nWARNING: no GPU runtime detected. This will still work but will be much slower.\n"
          "Go to Runtime -> Change runtime type -> GPU, then Runtime -> Restart and run all.")


## 1. Benchmark generator — `dataset_lib.py`, embedded verbatim (unchanged)

This is byte-for-byte the same generator used for every other row in the paper's Table 1 (Sparse PCA, Kernel PCA, t-SNE, the MLPRegressor autoencoder) and for the N=80-seed studies in Sections 5.4–6. Nothing here has been altered — DEC/IDEC will see exactly the same data every other baseline saw.

In [ ]:
"""
dataset_lib.py
================
Shared, seed-parameterized, mechanism-toggleable version of the hard
benchmark generator. This module factors out the logic in
generate_hard_dataset.py so that the interaction/robustness study
(interaction_robustness_study.py) can:

  (1) regenerate the dataset under many random seeds (to check that the
      headline numbers in results.json are not a seed=42 artifact), and
  (2) selectively DISABLE mechanism B (relational) and/or mechanism C
      (scale) to measure how much of the naive pipeline's damage to
      mechanism A (distance) is attributable to each mechanism alone,
      versus their combination -- i.e. to test for a super-additive
      interaction effect rather than just asserting one.

No behavior of the original generator changes when all mechanisms are
enabled and seed=42: this module is a strict refactor for reuse.
"""

import numpy as np
import pandas as pd

N_TOTAL = 8000
N_PER_CLASS = 2000
RELATIONAL_FRACTION = 0.12
SCALE_FRACTION = 0.04
RATIO = 2.3
DIST_CENTERS = [(4, 4), (-4, 4), (4, -4), (-4, -4)]
DIST_STD = 1.6


def make_distance_classes(rng, n_per_class, centers, std):
    feats, labels = [], []
    for i, c in enumerate(centers):
        pts = rng.normal(loc=c, scale=std, size=(n_per_class, 2))
        feats.append(pts)
        labels.append(np.full(n_per_class, i))
    return np.vstack(feats), np.concatenate(labels)


def inject_relational_subgroup(rng, n_total, fraction, ratio, noise_std=0.08):
    is_hidden = np.zeros(n_total, dtype=int)
    n_hidden = int(n_total * fraction)
    idx = rng.choice(n_total, size=n_hidden, replace=False)
    is_hidden[idx] = 1

    feature_3 = np.empty(n_total)
    feature_4 = np.empty(n_total)

    n_bg = n_total - n_hidden
    feature_3[is_hidden == 0] = rng.normal(0, 3.0, size=n_bg)
    feature_4[is_hidden == 0] = rng.normal(0, 3.0, size=n_bg)

    radius = rng.uniform(3, 9, size=n_hidden)
    sign = rng.choice([-1, 1], size=n_hidden)
    base = sign * radius
    feature_3[is_hidden == 1] = base + rng.normal(0, noise_std, size=n_hidden)
    feature_4[is_hidden == 1] = ratio * base + rng.normal(0, noise_std, size=n_hidden)

    return feature_3, feature_4, is_hidden


def inject_scale_subgroup(rng, n_total, fraction, host_std=3.0, minority_std=0.15):
    is_hidden = np.zeros(n_total, dtype=int)
    n_hidden = int(n_total * fraction)
    idx = rng.choice(n_total, size=n_hidden, replace=False)
    is_hidden[idx] = 1

    feature_5 = np.empty(n_total)
    feature_6 = np.empty(n_total)

    n_bg = n_total - n_hidden
    feature_5[is_hidden == 0] = rng.normal(0, host_std, size=n_bg)
    feature_6[is_hidden == 0] = rng.normal(0, host_std, size=n_bg)

    feature_5[is_hidden == 1] = rng.normal(0, minority_std, size=n_hidden)
    feature_6[is_hidden == 1] = rng.normal(0, minority_std, size=n_hidden)

    return feature_5, feature_6, is_hidden


def generate(seed, include_relational=True, include_scale=True, include_noise=True,
             relational_fraction=RELATIONAL_FRACTION, ratio=RATIO,
             scale_fraction=SCALE_FRACTION, dist_std=DIST_STD,
             scale_minority_std=0.15, n_per_class=N_PER_CLASS):
    """Generate one instance of the benchmark.

    Parameters
    ----------
    seed : int
        RNG seed. Using different seeds regenerates mechanism A (class
        centers/labels), mechanism B, and mechanism C independently but
        reproducibly, so every configuration at a given seed shares the
        same distance-mechanism draw ONLY when include_relational and
        include_scale are toggled with the same seed and the same rng
        call order is preserved (guaranteed here: A is always drawn
        first, then B, then C, then D, regardless of which are kept).
    relational_fraction, ratio, scale_fraction, dist_std, scale_minority_std : float
        Mechanism-strength knobs added for the parameter-space
        generalization study (Section 4.3). Defaults reproduce the
        original single-configuration benchmark exactly (no behavior
        change when called with no extra arguments -- verified by the
        original test in generate_hard_dataset.py).

    Returns
    -------
    observable : pd.DataFrame  (feature_1..feature_10, class_label)
    hidden : dict of np.ndarray  (hidden_relational_label, hidden_scale_label;
             all-zero arrays if the corresponding mechanism is disabled)
    """
    rng = np.random.default_rng(seed)

    ab, class_label = make_distance_classes(rng, n_per_class, DIST_CENTERS, dist_std)
    feature_1, feature_2 = ab[:, 0], ab[:, 1]
    n = len(class_label)
    perm = rng.permutation(n)
    feature_1, feature_2, class_label = feature_1[perm], feature_2[perm], class_label[perm]

    # Mechanism B is always DRAWN (to keep the RNG stream identical across
    # configs at a fixed seed) but its effect is discarded (replaced with
    # unstructured noise of the same marginal scale) when disabled, so that
    # turning B on/off is a clean ablation and not a confound of a
    # different random draw for feature_1/feature_2.
    feature_3, feature_4, hidden_relational = inject_relational_subgroup(
        rng, n, relational_fraction, ratio
    )
    if not include_relational:
        feature_3 = rng.normal(0, 3.0, size=n)
        feature_4 = rng.normal(0, 3.0, size=n)
        hidden_relational = np.zeros(n, dtype=int)

    feature_5, feature_6, hidden_scale = inject_scale_subgroup(
        rng, n, scale_fraction, minority_std=scale_minority_std
    )
    if not include_scale:
        feature_5 = rng.normal(0, 3.0, size=n)
        feature_6 = rng.normal(0, 3.0, size=n)
        hidden_scale = np.zeros(n, dtype=int)

    if include_noise:
        feature_7 = rng.normal(0, 2.0, size=n)
        feature_8 = rng.normal(0, 2.0, size=n)
        feature_9 = rng.normal(0, 2.0, size=n)
        feature_10 = rng.normal(0, 2.0, size=n)
    else:
        feature_7 = feature_8 = feature_9 = feature_10 = np.zeros(n)

    observable = pd.DataFrame({
        "feature_1": feature_1, "feature_2": feature_2,
        "feature_3": feature_3, "feature_4": feature_4,
        "feature_5": feature_5, "feature_6": feature_6,
        "feature_7": feature_7, "feature_8": feature_8,
        "feature_9": feature_9, "feature_10": feature_10,
        "class_label": class_label,
    })
    hidden = {
        "hidden_relational_label": hidden_relational,
        "hidden_scale_label": hidden_scale,
    }
    return observable, hidden


## 2. DEC / IDEC model

Standard formulation: an autoencoder is pretrained on reconstruction, its bottleneck is used to initialize K cluster centroids (K-Means), and a clustering layer then refines the embedding by minimizing the KL divergence between the soft (Student-t) cluster assignments `q` and a sharpened target distribution `p`, recomputed once per epoch until assignments stabilize (`tol` = fraction of points that change assignment between epochs).

- **DEC** (Xie et al. 2016): after pretraining, only the clustering (KL) loss is optimized; the decoder is frozen/unused.
- **IDEC** (Guo et al. 2017): the decoder stays active — the loss is `reconstruction_MSE + gamma * KL`, gamma = 0.1 by default — which the IDEC paper argues preserves local structure in the embedding better than DEC alone.

Both share one architecture class; `is_idec=True/False` switches the loss.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
import pandas as pd, time, os, json


class AutoencoderDEC(nn.Module):
    # input -> hidden_dims -> latent_dim (encoder), mirrored decoder,
    # plus a Student-t clustering layer over the latent space (alpha=1, per
    # van der Maaten & Hinton 2008 / Xie et al. 2016).

    def __init__(self, input_dim, hidden_dims=(500, 500, 2000), latent_dim=10,
                 n_clusters=4, alpha=1.0):
        super().__init__()
        dims = [input_dim] + list(hidden_dims) + [latent_dim]
        enc = []
        for i in range(len(dims) - 1):
            enc.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                enc.append(nn.ReLU())
        self.encoder = nn.Sequential(*enc)

        rdims = dims[::-1]
        dec = []
        for i in range(len(rdims) - 1):
            dec.append(nn.Linear(rdims[i], rdims[i + 1]))
            if i < len(rdims) - 2:
                dec.append(nn.ReLU())
        self.decoder = nn.Sequential(*dec)

        self.alpha = alpha
        self.cluster_centers = nn.Parameter(torch.zeros(n_clusters, latent_dim))

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        dist_sq = torch.sum((z.unsqueeze(1) - self.cluster_centers.unsqueeze(0)) ** 2, dim=2)
        q = 1.0 / (1.0 + dist_sq / self.alpha)
        q = q ** ((self.alpha + 1.0) / 2.0)
        q = q / torch.sum(q, dim=1, keepdim=True)
        return z, x_recon, q


def target_distribution(q):
    weight = q ** 2 / torch.sum(q, dim=0)
    return (weight.t() / torch.sum(weight, dim=1)).t()


## 3. Training routines (pretrain, cluster-center init, DEC/IDEC self-training)

In [ ]:
def pretrain(model, X, epochs=50, lr=1e-3, batch_size=256, device="cpu"):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = X.shape[0]
    model.train()
    for ep in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            xb = X[idx].to(device)
            opt.zero_grad()
            z = model.encode(xb)
            xr = model.decoder(z)
            loss = F.mse_loss(xr, xb)
            loss.backward()
            opt.step()
    return model


def init_clusters(model, X, n_clusters, device="cpu", seed=0):
    model.eval()
    with torch.no_grad():
        z = model.encode(X.to(device)).cpu().numpy()
    km = KMeans(n_clusters=n_clusters, n_init=20, random_state=seed).fit(z)
    model.cluster_centers.data = torch.tensor(km.cluster_centers_, dtype=torch.float32).to(device)
    return model


def train_dec_idec(model, X, is_idec, gamma=0.1, max_epochs=60, tol=1e-3,
                    lr=1e-4, batch_size=256, device="cpu"):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = X.shape[0]
    y_pred_last = None
    epochs_run = 0
    for ep in range(max_epochs):
        epochs_run = ep + 1
        model.eval()
        with torch.no_grad():
            z, xr, q = model(X.to(device))
            p = target_distribution(q).detach()
            y_pred = q.argmax(1).cpu().numpy()
        if y_pred_last is not None:
            delta = float(np.mean(y_pred != y_pred_last))
            if delta < tol:
                break
        y_pred_last = y_pred

        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            xb = X[idx].to(device)
            pb = p[idx]
            opt.zero_grad()
            z, xr, q = model(xb)
            loss_kl = F.kl_div(torch.log(q + 1e-10), pb, reduction="batchmean")
            if is_idec:
                loss_recon = F.mse_loss(xr, xb)
                loss = loss_recon + gamma * loss_kl
            else:
                loss = loss_kl
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        z, xr, q = model(X.to(device))
    y_pred = q.argmax(1).cpu().numpy()
    return model, z.cpu().numpy(), y_pred, epochs_run


## 4. Evaluation — identical protocol to Table 1's other rows

`cluster_and_score` is the same function (same k-grid, same Silhouette-based k selection, same three ARI columns) used in `deep_clustering_baseline.py` for the existing autoencoder row, applied here to DEC/IDEC's final latent embedding — for direct comparability. We additionally report DEC/IDEC's own native k=4 soft-assignment ARI (`native_ari_*`), since unlike the other rows, DEC/IDEC produce a clustering directly and a post-hoc K-Means step is not actually part of the method.

In [ ]:
def cluster_and_score(embedding, class_label, hidden_relational, hidden_scale, k_grid=(2, 3, 4, 5, 6)):
    sil_by_k, ari_by_k, models = {}, {}, {}
    for k in k_grid:
        km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(embedding)
        sil_by_k[k] = float(silhouette_score(embedding, km.labels_))
        ari_by_k[k] = float(adjusted_rand_score(class_label, km.labels_))
        models[k] = km
    best_k = max(sil_by_k, key=sil_by_k.get)
    return {
        "best_k_by_silhouette": int(best_k),
        "ari_at_silhouette_best_k": ari_by_k[best_k],
        "ari_at_k4": ari_by_k[4],
        "ari_relational_hidden_at_k4": float(adjusted_rand_score(hidden_relational, models[4].labels_)),
        "ari_scale_hidden_at_k4": float(adjusted_rand_score(hidden_scale, models[4].labels_)),
    }


def run_one(seed, method, latent_dim, n_clusters=4, device=DEVICE,
            pretrain_epochs=50, cluster_epochs=60, hidden_dims=(500, 500, 2000)):
    obs, hidden = generate(seed)
    class_label = obs["class_label"].values
    hidden_relational = hidden["hidden_relational_label"]
    hidden_scale = hidden["hidden_scale_label"]
    feature_cols = [c for c in obs.columns if c != "class_label"]
    Xs = StandardScaler().fit_transform(obs[feature_cols].values)
    Xt = torch.tensor(Xs, dtype=torch.float32)

    torch.manual_seed(seed)
    np.random.seed(seed)

    model = AutoencoderDEC(input_dim=Xs.shape[1], hidden_dims=hidden_dims,
                            latent_dim=latent_dim, n_clusters=n_clusters)
    model = pretrain(model, Xt, epochs=pretrain_epochs, device=device)
    model = init_clusters(model, Xt, n_clusters, device=device, seed=seed)
    model, z_np, y_pred, epochs_run = train_dec_idec(
        model, Xt, is_idec=(method == "idec"), max_epochs=cluster_epochs, device=device
    )

    native_a = float(adjusted_rand_score(class_label, y_pred))
    native_b = float(adjusted_rand_score(hidden_relational, y_pred))
    native_c = float(adjusted_rand_score(hidden_scale, y_pred))
    posthoc = cluster_and_score(z_np, class_label, hidden_relational, hidden_scale)

    return dict(seed=seed, method=method, latent_dim=latent_dim, epochs_run=epochs_run,
                native_ari_a=native_a, native_ari_b=native_b, native_ari_c=native_c,
                posthoc_ari_at_k4=posthoc["ari_at_k4"],
                posthoc_ari_relational_at_k4=posthoc["ari_relational_hidden_at_k4"],
                posthoc_ari_scale_at_k4=posthoc["ari_scale_hidden_at_k4"],
                posthoc_best_k_by_silhouette=posthoc["best_k_by_silhouette"],
                posthoc_ari_at_silhouette_best_k=posthoc["ari_at_silhouette_best_k"])


## 5. Headline seed=42 runs — comparability with Table 1's existing rows

Two latent sizes: `latent_dim=2` (matches the existing autoencoder row's bottleneck exactly) and `latent_dim=10` (the 'wider architecture' the paper's Future Work calls for). ~2–5 min per run on a T4 GPU.

In [ ]:
headline_rows = []
t0 = time.time()
for method in ["dec", "idec"]:
    for latent_dim in [2, 10]:
        # latent_dim=2 case mirrors the paper's original MLPRegressor
        # architecture exactly: input-16-2-16-input (ONE hidden layer of
        # 16 units each side of the bottleneck, not two).
        hidden_dims = (16,) if latent_dim == 2 else (500, 500, 2000)
        r = run_one(seed=42, method=method, latent_dim=latent_dim, hidden_dims=hidden_dims)
        headline_rows.append(r)
        print(f"[headline] method={method:4s} latent={latent_dim:2d}  "
              f"ARI_A={r['native_ari_a']:.3f}  ARI_B={r['native_ari_b']:.3f}  "
              f"ARI_C={r['native_ari_c']:.3f}   (t={time.time()-t0:.0f}s)")

headline_df = pd.DataFrame(headline_rows)
headline_df


## 6. N = 80-seed sweep (seeds 1000–1079) — same convention as Sections 5.4/5.5/5.6/6

Run at `latent_dim=10` (the wider, recommended architecture) for both DEC and IDEC. This cell is resumable: it skips (seed, method) pairs already present in the CSV, so if Colab disconnects you can just re-run this cell.

**Runtime budget:** 160 total (seed, method) runs. On a T4 this is roughly 3–8 hours depending on convergence speed. If you need a faster first look, drop `SEEDS` to `range(1000, 1020)` (N=20, matching the paper's original — since-corrected — autoencoder sweep size) for a quick sanity check before committing to the full N=80 run.

In [ ]:
SEEDS = range(1000, 1080)   # N = 80, paper's standing convention (Sections 5.4-6)
METHODS = ["dec", "idec"]
LATENT_DIM = 10
HIDDEN_DIMS = (500, 500, 2000)

RAW_PATH = "/content/dec_idec_multiseed_raw.csv"
rows = pd.read_csv(RAW_PATH).to_dict("records") if os.path.exists(RAW_PATH) else []
done = {(r["seed"], r["method"]) for r in rows}

t0 = time.time()
for seed in SEEDS:
    for method in METHODS:
        if (seed, method) in done:
            continue
        r = run_one(seed=seed, method=method, latent_dim=LATENT_DIM, hidden_dims=HIDDEN_DIMS)
        rows.append(r)
        print(f"seed={seed} method={method:4s}  ARI_A={r['native_ari_a']:.3f} "
              f"ARI_B={r['native_ari_b']:.3f} ARI_C={r['native_ari_c']:.3f}  "
              f"t={time.time()-t0:.0f}s")
        pd.DataFrame(rows).to_csv(RAW_PATH, index=False)

multiseed_df = pd.DataFrame(rows)
multiseed_df.tail()


## 7. Summary — formatted to slot directly into Table 1 / Section 5.3

In [ ]:
RECOVERY_THRESHOLD = 0.3  # same threshold used for the existing autoencoder row

def summarize(df, method):
    d = df[df.method == method]
    a_hi = d.native_ari_a > RECOVERY_THRESHOLD
    b_hi = d.native_ari_b > RECOVERY_THRESHOLD
    c_hi = d.native_ari_c > RECOVERY_THRESHOLD
    return {
        "n_seeds": int(len(d)),
        "native_ari_a": {"mean": float(d.native_ari_a.mean()), "sd": float(d.native_ari_a.std()),
                          "min": float(d.native_ari_a.min()), "max": float(d.native_ari_a.max())},
        "native_ari_b": {"mean": float(d.native_ari_b.mean()), "sd": float(d.native_ari_b.std()),
                          "min": float(d.native_ari_b.min()), "max": float(d.native_ari_b.max())},
        "native_ari_c": {"mean": float(d.native_ari_c.mean()), "sd": float(d.native_ari_c.std()),
                          "min": float(d.native_ari_c.min()), "max": float(d.native_ari_c.max())},
        "frac_seeds_A_recovered": float(a_hi.mean()),
        "frac_seeds_B_recovered": float(b_hi.mean()),
        "frac_seeds_C_recovered": float(c_hi.mean()),
        "frac_seeds_both_AB_recovered": float((a_hi & b_hi).mean()),
    }

summary = {"headline_seed42": headline_df.to_dict("records"),
           "multiseed_n80_latent10": {m: summarize(multiseed_df, m) for m in METHODS}}

with open("/content/dec_idec_results.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)

print(json.dumps(summary["multiseed_n80_latent10"], indent=2))


## 8. Package results for download — send this back

In [ ]:
import shutil
shutil.make_archive("/content/dec_idec_results", "zip", "/content",
                     logger=None)
# (zips everything in /content; simplest reliable approach in a fresh runtime)
from google.colab import files
files.download("/content/dec_idec_results.zip")
print("Downloaded dec_idec_results.zip - send this file back, or just paste the",
      "printed summary table above.")


### What to send back

Either:
1. The downloaded `dec_idec_results.zip` (contains `dec_idec_results.json` — the summary — and `dec_idec_multiseed_raw.csv` — every individual seed's numbers), **or**
2. Just paste the printed JSON summary from cell 7/9 back into the chat.

Either is enough to update Table 1 (new `DEC` and `IDEC` rows), Section 5.3's discussion, and to close out the corresponding Future Work / Limitations bullets.